# Part 1: Extract Tumor-Focused Patches from BraTS2020

## This notebook is in: `notebooks/` folder
## Saves patches to: `patches/` folder (parallel to notebooks/)

```
Project/
├── notebooks/  ← You are here
│   └── 01_extract_patches.ipynb
├── patches/  ← Patches will be saved here
│   ├── train/
│   │   ├── images/
│   │   └── masks/
│   └── val/
│       ├── images/
│       └── masks/
├── BraTS2020_training_data/
└── BraTS2020_ValidationData/
```

---

In [ ]:
# Cell 1: Imports

import os
import numpy as np
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
import random

random.seed(42)
np.random.seed(42)

print("="*80)
print(" "*20 + "BRATS2020 PATCH EXTRACTION")
print("="*80)

In [ ]:
# Cell 2: Configuration

class Config:
    # This notebook is in notebooks/ folder
    # Go UP one level (..) to get to project root
    PROJECT_ROOT = Path('..')  
    
    # Input data (at project root level)
    TRAIN_DATA_DIR = PROJECT_ROOT / 'BraTS2020_training_data' / 'MICCAI_BraTS2020_TrainingData'
    VAL_DATA_DIR = PROJECT_ROOT / 'BraTS2020_ValidationData' / 'MICCAI_BraTS2020_ValidationData'
    
    # Output: patches/ folder at project root (parallel to notebooks/)
    OUTPUT_DIR = PROJECT_ROOT / 'patches'
    
    # Patch settings
    PATCH_SIZE = 64
    PATCHES_PER_VOLUME_WITH_TUMOR = 20
    PATCHES_PER_VOLUME_NO_TUMOR = 5
    TUMOR_PATCH_RATIO = 0.8
    
config = Config()

# Detailed path verification
print(f"\n🔍 PATH VERIFICATION:")
print(f"="*80)
print(f"Current directory: {Path.cwd()}")
print(f"Project root: {config.PROJECT_ROOT.resolve()}")
print(f"\nInput Data:")
print(f"  Train: {config.TRAIN_DATA_DIR.resolve()}")
print(f"         Exists: {config.TRAIN_DATA_DIR.exists()}")
print(f"  Val:   {config.VAL_DATA_DIR.resolve()}")
print(f"         Exists: {config.VAL_DATA_DIR.exists()}")
print(f"\nOutput Directory:")
print(f"  {config.OUTPUT_DIR.resolve()}")
print(f"  Exists: {config.OUTPUT_DIR.exists()}")

# Check specific output folders
print(f"\n📁 OUTPUT FOLDERS:")
print(f"="*80)
folders = [
    config.OUTPUT_DIR / 'train' / 'images',
    config.OUTPUT_DIR / 'train' / 'masks',
    config.OUTPUT_DIR / 'val' / 'images',
    config.OUTPUT_DIR / 'val' / 'masks'
]
for folder in folders:
    print(f"  {folder.resolve()}")
    print(f"    Exists: {folder.exists()}")
    if folder.exists():
        num_files = len(list(folder.glob('*.npy')))
        print(f"    Current files: {num_files}")

# Create folders if missing
print(f"\n⚙️  Ensuring all folders exist...")
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
print(f"  ✓ Done")

print(f"\n📋 CONFIGURATION:")
print(f"="*80)
print(f"  Patch size: {config.PATCH_SIZE}³")
print(f"  Patches per volume: {config.PATCHES_PER_VOLUME_WITH_TUMOR}")
print(f"  Tumor patch ratio: {config.TUMOR_PATCH_RATIO*100}%")
print(f"="*80)

In [ ]:
# Cell 3: Patch Extraction Functions

def normalize_modality(img):
    brain_mask = img > 0
    if brain_mask.sum() > 0:
        mean = img[brain_mask].mean()
        std = img[brain_mask].std() + 1e-8
        img = (img - mean) / std
        img = np.clip(img, -5, 5)
    return img.astype(np.float32)

def extract_random_patch(image, mask, patch_size, tumor_focused=True):
    D, H, W = image.shape[1:]
    
    if tumor_focused:
        tumor_coords = np.argwhere(mask > 0)
        if len(tumor_coords) > 0:
            center = tumor_coords[np.random.randint(len(tumor_coords))]
            d_start = max(0, min(center[0] - patch_size//2, D - patch_size))
            h_start = max(0, min(center[1] - patch_size//2, H - patch_size))
            w_start = max(0, min(center[2] - patch_size//2, W - patch_size))
        else:
            tumor_focused = False
    
    if not tumor_focused:
        d_start = np.random.randint(0, max(1, D - patch_size + 1))
        h_start = np.random.randint(0, max(1, H - patch_size + 1))
        w_start = np.random.randint(0, max(1, W - patch_size + 1))
    
    image_patch = image[:, d_start:d_start+patch_size, h_start:h_start+patch_size, w_start:w_start+patch_size]
    mask_patch = mask[d_start:d_start+patch_size, h_start:h_start+patch_size, w_start:w_start+patch_size]
    
    if image_patch.shape[1:] != (patch_size, patch_size, patch_size):
        pad_d = patch_size - image_patch.shape[1]
        pad_h = patch_size - image_patch.shape[2]
        pad_w = patch_size - image_patch.shape[3]
        image_patch = np.pad(image_patch, ((0,0),(0,pad_d),(0,pad_h),(0,pad_w)))
        mask_patch = np.pad(mask_patch, ((0,pad_d),(0,pad_h),(0,pad_w)))
    
    return image_patch, mask_patch

def process_volume(patient_dir, patch_size, num_patches, tumor_ratio):
    """Extract ONLY tumor-containing patches."""
    patches_images = []
    patches_masks = []
    
    try:
        modalities = []
        for mod in ['t1', 't1ce', 't2', 'flair']:
            files = list(patient_dir.glob(f'*_{mod}.nii*'))
            if len(files) == 0:
                return [], []
            img = nib.load(str(files[0])).get_fdata().astype(np.float32)
            img = normalize_modality(img)
            modalities.append(img)
        image = np.stack(modalities, axis=0)
        
        seg_files = list(patient_dir.glob('*_seg.nii*'))
        if len(seg_files) > 0:
            mask = nib.load(str(seg_files[0])).get_fdata().astype(np.int64)
            mask[mask == 4] = 3
        else:
            mask = np.zeros(image.shape[1:], dtype=np.int64)
        
        has_tumor = (mask > 0).sum() > 100
        
        if not has_tumor:
            return [], []  # SKIP volumes without tumors!
        
        # Extract ONLY tumor patches
        for _ in range(num_patches):
            img_patch, mask_patch = extract_random_patch(image, mask, patch_size, tumor_focused=True)
            
            # VERIFY patch has tumor
            if (mask_patch > 0).sum() > 50:  # At least 50 tumor voxels
                patches_images.append(img_patch)
                patches_masks.append(mask_patch)
    
    except Exception as e:
        return [], []
    
    return patches_images, patches_masks

print("✓ Functions loaded")

In [ ]:
# Cell 4: Extract Training Patches

print("\n" + "="*80)
print("EXTRACTING TRAINING PATCHES")
print("="*80)

if not config.TRAIN_DATA_DIR.exists():
    print(f"❌ ERROR: Training data not found!")
    print(f"   Looking for: {config.TRAIN_DATA_DIR.resolve()}")
else:
    patient_dirs = sorted([d for d in config.TRAIN_DATA_DIR.iterdir() if d.is_dir()])
    print(f"Found {len(patient_dirs)} training volumes")
    print(f"Saving to: {config.OUTPUT_DIR.resolve() / 'train'}\n")
    
    patch_counter = 0
    
    for patient_dir in tqdm(patient_dirs, desc="Processing"):
        patches_img, patches_msk = process_volume(
            patient_dir, config.PATCH_SIZE, 
            config.PATCHES_PER_VOLUME_WITH_TUMOR, config.TUMOR_PATCH_RATIO
        )
        
        for img, msk in zip(patches_img, patches_msk):
            img_path = config.OUTPUT_DIR / 'train' / 'images' / f'patch_{patch_counter:05d}.npy'
            msk_path = config.OUTPUT_DIR / 'train' / 'masks' / f'patch_{patch_counter:05d}.npy'
            np.save(img_path, img)
            np.save(msk_path, msk)
            patch_counter += 1
    
    print(f"\n✓ Saved {patch_counter} training patches")
    
    # Verify
    actual = len(list((config.OUTPUT_DIR / 'train' / 'images').glob('*.npy')))
    print(f"  Files on disk: {actual}")
    if actual == patch_counter:
        print(f"  ✅ All saved successfully!")
    else:
        print(f"  ⚠️  Mismatch: created {patch_counter}, found {actual}")

In [ ]:
# Cell 5: Extract Validation Patches

print("\n" + "="*80)
print("EXTRACTING VALIDATION PATCHES")
print("="*80)

if not config.VAL_DATA_DIR.exists():
    print(f"❌ ERROR: Validation data not found!")
    print(f"   Looking for: {config.VAL_DATA_DIR.resolve()}")
else:
    patient_dirs = sorted([d for d in config.VAL_DATA_DIR.iterdir() if d.is_dir()])
    print(f"Found {len(patient_dirs)} validation volumes")
    print(f"Saving to: {config.OUTPUT_DIR.resolve() / 'val'}\n")
    
    patch_counter = 0
    
    for patient_dir in tqdm(patient_dirs, desc="Processing"):
        patches_img, patches_msk = process_volume(
            patient_dir, config.PATCH_SIZE, 
            config.PATCHES_PER_VOLUME_NO_TUMOR, 0.5
        )
        
        for img, msk in zip(patches_img, patches_msk):
            img_path = config.OUTPUT_DIR / 'val' / 'images' / f'patch_{patch_counter:05d}.npy'
            msk_path = config.OUTPUT_DIR / 'val' / 'masks' / f'patch_{patch_counter:05d}.npy'
            np.save(img_path, img)
            np.save(msk_path, msk)
            patch_counter += 1
    
    print(f"\n✓ Saved {patch_counter} validation patches")
    
    # Verify
    actual = len(list((config.OUTPUT_DIR / 'val' / 'images').glob('*.npy')))
    print(f"  Files on disk: {actual}")
    if actual == patch_counter:
        print(f"  ✅ All saved successfully!")
    else:
        print(f"  ⚠️  Mismatch: created {patch_counter}, found {actual}")

In [ ]:
# Cell 6: Final Summary

print("\n" + "="*80)
print("EXTRACTION COMPLETE")
print("="*80)

train_imgs = len(list((config.OUTPUT_DIR / 'train' / 'images').glob('*.npy')))
train_msks = len(list((config.OUTPUT_DIR / 'train' / 'masks').glob('*.npy')))
val_imgs = len(list((config.OUTPUT_DIR / 'val' / 'images').glob('*.npy')))
val_msks = len(list((config.OUTPUT_DIR / 'val' / 'masks').glob('*.npy')))

print(f"\n📊 FINAL COUNTS:")
print(f"  Training:   {train_imgs} images, {train_msks} masks")
print(f"  Validation: {val_imgs} images, {val_msks} masks")
print(f"  Total:      {train_imgs + val_imgs} patches")

if train_imgs > 0 and val_imgs > 0 and train_imgs == train_msks and val_imgs == val_msks:
    print(f"\n✅ SUCCESS!\n")
    print(f"📁 Patches saved to:")
    print(f"   {(config.OUTPUT_DIR / 'train').resolve()}")
    print(f"   {(config.OUTPUT_DIR / 'val').resolve()}")
    
    sample = np.load(config.OUTPUT_DIR / 'train' / 'images' / 'patch_00000.npy')
    sample_mask = np.load(config.OUTPUT_DIR / 'train' / 'masks' / 'patch_00000.npy')
    print(f"\n🔍 Sample verification:")
    print(f"   Image shape: {sample.shape} (expected: (4, 64, 64, 64))")
    print(f"   Mask shape: {sample_mask.shape} (expected: (64, 64, 64))")
    print(f"   Mask labels: {np.unique(sample_mask)}")
    
    if sample.shape == (4, 64, 64, 64) and sample_mask.shape == (64, 64, 64):
        print(f"   ✅ Shapes correct!")
    
    print(f"\n🎯 Next step: Run 02_train_swin.ipynb")
else:
    print(f"\n❌ ERROR: Something went wrong!")
    if train_imgs == 0:
        print(f"   No training patches created")
    if val_imgs == 0:
        print(f"   No validation patches created")
    if train_imgs != train_msks:
        print(f"   Training images/masks mismatch")
    if val_imgs != val_msks:
        print(f"   Validation images/masks mismatch")

print("="*80)